# 31 - Data Cleaning Notebook

This notebook documents the full data cleaning workflow for the Sales Pipeline Analysis project. 
It mirrors the logic implemented in `clean_data.py`, but presents it in a step-by-step, 
human-readable format with explanations, previews, and validation checks.

The goal is to:
- Understand the raw data
- Apply cleaning transformations
- Validate the cleaned outputs
- Export cleaned CSVs for downstream SQL + analysis


In [26]:
import pandas as pd
from pathlib import Path

raw_path = Path(r"...\data\raw")
clean_path = Path(r"...\data\cleaned")


## Load Raw Data

First we'll load up the raw CSV files exactly as they exist in the `/data/raw/` directory.


In [27]:
# Load up all raw CSV files
accounts_raw = pd.read_csv(raw_path / "accounts_raw.csv")
products_raw = pd.read_csv(raw_path / "products_raw.csv")
teams_raw = pd.read_csv(raw_path / "teams_raw.csv")
pipeline_raw = pd.read_csv(raw_path / "pipeline_raw.csv")

# View the first few rows

pipeline_raw.head()


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0


In [28]:
accounts_raw.head()

,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,NaN
1,Betasoloin,medical,1999,251.41,495,United States,NaN
2,Betatech,medical,1986,647.18,1185,Kenya,NaN
3,Bioholding,medical,2012,587.34,1356,Philipines,NaN
4,Bioplex,medical,1991,326.82,1016,United States,NaN


In [29]:
products_raw.head()

,product,series,sales_price
0,GTX Basic,GTX,550
1,GTX Pro,GTX,4821
2,MG Special,MG,55
3,MG Advanced,MG,3393
4,GTX Plus Pro,GTX,5482


In [30]:
teams_raw.head()

,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central


## Cleaning Steps

Now we can apply the same transformations implemented in `clean_data.py`, including:

- Standardizing column names
- Parsing dates
- Handling missing values
- Normalizing categorical fields


In [8]:
# Get rid of any funky whitespace we might have going on.
# Ensures everything is just lowercase string formatting for consistency's sake.
# This makes it so column names between tables can match up easier for PostgreSQL.
accounts_raw.columns = accounts_raw.columns.str.lower().str.strip()

products_raw.columns = products_raw.columns.str.lower().str.strip()

pipeline_raw.columns = pipeline_raw.columns.str.lower().str.strip()

teams_raw.columns = teams_raw.columns.str.lower().str.strip()



In [9]:
pipeline_raw['product'] = pipeline_raw['product'].replace({'GTXPro': 'Gtx Pro'})
# Gets rid of that free-floating error where some GTX Pro's are called different things.

In [10]:
# Defining a block, here -- clean_text(x).
# We'll apply this "clean text" function to our tables to normalize column names.
def clean_text(x):
    if pd.isna(x):
        return x
    return x.strip().title()

In [11]:
# Applying.
pipeline_raw['product'] = pipeline_raw['product'].apply(clean_text)
products_raw['product'] = products_raw['product'].apply(clean_text)

In [12]:
# This is that same line as earlier.
# Not sure why to be frank, but if we get rid of one or the other, it doesn't work.
# Probably just missing something, but if something works, something works.
# That being said, even though it's redundant, we're leaving it.
pipeline_raw['product'] = pipeline_raw['product'].replace({'GTXPro': 'Gtx Pro'})

In [13]:
# Back to applying clean text.
accounts_raw['account'] = accounts_raw['account'].apply(clean_text)
pipeline_raw['account'] = pipeline_raw['account'].apply(clean_text)

# This is supposed to show us any "bad" account names.
set(pipeline_raw['account']) - set(accounts_raw['account'])

# Back to applying clean text.
teams_raw['sales_agent'] = teams_raw['sales_agent'].apply(clean_text)
pipeline_raw['sales_agent'] = pipeline_raw['sales_agent'].apply(clean_text)

# Back to what's supposed to show us any "bad" sales agent names.
set(pipeline_raw['sales_agent']) - set(teams_raw['sales_agent'])

# For the record, applying the clean text is going to let our foreign keys link up in PostgreSQL.

set()

In [14]:
# Converting to datetime. Super necessary.
pipeline_raw['engage_date'] = pd.to_datetime(pipeline_raw['engage_date'], errors='coerce')
pipeline_raw['close_date'] = pd.to_datetime(pipeline_raw['close_date'], errors='coerce')

In [15]:
# Converting revenue, value, etc. to numeric type.
pipeline_raw['close_value'] = pd.to_numeric(pipeline_raw['close_value'], errors='coerce')
products_raw['sales_price'] = pd.to_numeric(products_raw['sales_price'], errors='coerce')
accounts_raw['revenue'] = pd.to_numeric(accounts_raw['revenue'], errors='coerce')
accounts_raw['employees'] = pd.to_numeric(accounts_raw['employees'], errors='coerce')

In [16]:
# Handling NAs in the 'subsidiary_of' column.
accounts_raw['subsidiary_of'] = accounts_raw['subsidiary_of'].replace("", pd.NA)

# More redundancy. 
pipeline_raw['engage_date'] = pd.to_datetime(pipeline_raw['engage_date'], errors='coerce')
pipeline_raw['close_date'] = pd.to_datetime(pipeline_raw['close_date'], errors='coerce')
pipeline_raw['close_value'] = pd.to_numeric(pipeline_raw['close_value'], errors='coerce')

# Handling the other NAs.
accounts_raw['account'] = accounts_raw['account'].replace("", pd.NA)
pipeline_raw['close_date'] = pipeline_raw['close_date'].replace("", pd.NA)
pipeline_raw['engage_date'] = pipeline_raw['engage_date'].replace("", pd.NA)
pipeline_raw['close_value'] = pipeline_raw['close_value'].replace("", pd.NA)

## Validation

Now we can re-examine the data overall and make sure that things are looking good.

In [17]:
# Accounts first:
accounts_raw.info()
accounts_raw.head()
accounts_raw.isna().sum()

# Pipeline next:
pipeline_raw.info()
pipeline_raw.head()
pipeline_raw.isna().sum()

# Teams:
teams_raw.info()
teams_raw.head()
teams_raw.isna().sum()

# Products:
products_raw.info()
products_raw.head()
products_raw.isna().sum()


<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     str    
 1   sector            85 non-null     str    
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     str    
 6   subsidiary_of     15 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 4.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  8800 non-null   str           
 1   sales_agent     8800 non-null   str           
 2   product         8800 non-null   str           
 3   account         7375 non-null   str           
 4   deal_stage      

product        0
series         0
sales_price    0
dtype: int64

# Export Cleaned Data
Finally, we can save the cleaned data into `/data/cleaned/` for use in SQL and Exploratory Data Analysis (EDA).

In [19]:
accounts_raw.to_csv(clean_path / "accounts_clean.csv", index=False)
products_raw.to_csv(clean_path / "products_clean.csv", index=False)
teams_raw.to_csv(clean_path / "teams_clean.csv", index=False)
pipeline_raw.to_csv(clean_path / "pipeline_clean.csv", index=False)